In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import os

# 输入和输出文件夹路径
input_folder = r""        # 替换为你的输入文件夹路径
output_folder = r""   # 替换为你的输出文件夹路径

# 自定义颜色
custom_colors = np.array([
    [255, 215, 0],     # 黄色（水和阴影）
    [139, 69, 19],     # 棕色（建筑）
    [176, 176, 176],   # 银色（道路）
    [255, 160, 122],   # 浅橙色（土地）
    [128, 128, 0],     # 橄榄色（植被）
], dtype=np.uint8)

# 创建输出文件夹
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 聚类数量
num_clusters = len(custom_colors)

# 遍历输入文件夹中的图像
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif")):
        img_path = os.path.join(input_folder, filename)
        print(f"处理图像: {img_path}")

        image = cv2.imread(img_path)
        if image is None:
            print(f"无法读取图像 {img_path}")
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        h, w, c = image.shape
        image_2d = image.reshape(-1, 3)

        kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
        kmeans.fit(image_2d)
        labels = kmeans.labels_

        if len(custom_colors) < num_clusters:
            extra_colors = np.random.randint(0, 255, size=(num_clusters - len(custom_colors), 3))
            custom_colors = np.vstack([custom_colors, extra_colors])

        segmented_image = custom_colors[labels]
        segmented_image = segmented_image.reshape(h, w, 3)

        # 使用原文件名（去掉扩展名），统一保存为 .png
        base_name = os.path.splitext(filename)[0]
        save_path = os.path.join(output_folder, f"seg_{base_name}.png")
        cv2.imwrite(save_path, cv2.cvtColor(segmented_image, cv2.COLOR_RGB2BGR))
        print(f"保存成功: {save_path}")

print("所有图像处理完成。")


In [ ]:
import cv2
import numpy as np
import os

# 输入和输出文件夹路径
input_folder = r""        # 聚类彩色图像的文件夹
output_folder = r""        # 转换后灰度图像的文件夹

# 如果输出文件夹不存在，则创建
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 定义颜色到灰度值的映射（BGR格式）
color_value_map = {
    (255, 215, 0): 0,       # 水和阴影
    (139, 69, 19): 255,     # 建筑
    (176, 176, 176): 99,    # 道路
    (255, 160, 122): 99,    # 土壤
    (128, 128, 0): 149      # 植被
}

# 颜色容忍度
TOLERANCE = 10

# 遍历所有图像文件
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif")):
        input_path = os.path.join(input_folder, filename)
        print(f"处理图像: {input_path}")
        
        clustered_img = cv2.imread(input_path)
        if clustered_img is None:
            print(f"读取失败: {input_path}")
            continue

        # 初始化灰度输出图像
        output = np.zeros(clustered_img.shape[:2], dtype=np.uint8)

        for color_bgr, value in color_value_map.items():
            color = np.array(color_bgr)
            lower = np.clip(color - TOLERANCE, 0, 255)
            upper = np.clip(color + TOLERANCE, 0, 255)

            mask = cv2.inRange(clustered_img, lower, upper)
            output[mask > 0] = value

        # 构造输出文件名（强制为 .png）
        base_name = os.path.splitext(filename)[0]
        output_path = os.path.join(output_folder, f"gray_{base_name}.png")
        cv2.imwrite(output_path, output)
        print(f"保存灰度图: {output_path}")

print("所有图像转换完成。")